<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/Runnable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
import datasets  # 1. Import the base datasets module first
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

# =========================================================================
# THE FIX: Force the datasets library to ignore torchvision video utilities
# =========================================================================
datasets.config.TORCHVISION_AVAILABLE = False

def main():
    # ==========================================
    # 1. HARDWARE CONSTRAINTS & HYPERPARAMETERS
    # ==========================================
    device = torch.device('cpu') # Enforce CPU constraint
    BATCH_SIZE = 4               # Physical batch size (fits comfortably in 8GB RAM)
    ACCUMULATE_STEPS = 4         # Effective batch size = 16 (4 * 4)
    EPOCHS = 1                   # 1 epoch for quick demonstration

    print(f"--- Starting CPU-Optimized Training Pipeline ---")
    print(f"Device: {device} | Batch Size: {BATCH_SIZE} | Accumulation Steps: {ACCUMULATE_STEPS}")

    # ==========================================
    # 2. GENERATE IN-MEMORY DATASET
    # ==========================================
    print("\nInitializing Tokenizer and creating local synthetic dataset...")
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

    synthetic_data = {
        "text": [
            "This movie was absolutely brilliant! The acting was sublime.",
            "Horrible film. A complete waste of time and money.",
            "I loved the cinematography, but the plot was a bit weak.",
            "Utter trash. Worst directorial choice ever made.",
            "An absolute masterpiece. I watched it three times!",
            "Incredibly boring. Fell asleep halfway through the show."
        ] * 20,  # Generates 120 balanced samples for a quick local demo
        "label": [1, 0, 1, 0, 1, 0] * 20
    }

    raw_dataset = Dataset.from_dict(synthetic_data)

    def preprocess_function(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

    # Tokenize and format cleanly for PyTorch Tensors
    tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # ==========================================
    # 3. LOAD MODEL & APPLY LAYER FREEZING
    # ==========================================
    print("\nLoading DistilBERT Model...")
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

    print("Applying Layer Freezing to bottom 4 layers...")
    for param in model.distilbert.embeddings.parameters():
        param.requires_grad = False

    for layer in model.distilbert.transformer.layer[:4]:
        for param in layer.parameters():
            param.requires_grad = False

    # Calculate parameter optimization metrics
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.1f}%)")

    model.to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

    # ==========================================
    # 4. TRAINING LOOP WITH GRADIENT ACCUMULATION
    # ==========================================
    print("\nStarting Training Loop...")
    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0
        optimizer.zero_grad()

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(progress_bar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            # Mathematical normalization for gradient accumulation
            loss = loss / ACCUMULATE_STEPS
            total_loss += loss.item()

            # Backward pass
            loss.backward()

            # Step the optimizer only after gathering gradients from 'ACCUMULATE_STEPS'
            if (step + 1) % ACCUMULATE_STEPS == 0 or (step + 1) == len(dataloader):
                optimizer.step()
                optimizer.zero_grad()

            progress_bar.set_postfix({'loss': f"{loss.item() * ACCUMULATE_STEPS:.4f}"})

    print("\n CPU Training Demonstration Complete and Error-Free!")

if __name__ == "__main__":
    main()